In [0]:
%pip install shap==0.51.0
%pip install xgboost==3.2.0


In [0]:
%restart_python

In [0]:
# Importing libraries

import os
import numpy as np
import pandas as pd
import mlflow
import mlflow.xgboost
import shap
from pyspark.sql.functions import col
from pyspark.ml.functions import vector_to_array
from mlflow.tracking import MlflowClient

os.environ['MLFLOW_DFS_TMP'] = '/Volumes/workspace/ml_layer/mlflow_tmp'

def ensure_catalog():
    spark.sql("USE CATALOG workspace")
    spark.sql("USE DATABASE ml_layer")

ensure_catalog()

TXN_FEATURE_NAMES = [
    "log_amount", "amount_balance_ratio", "transaction_hour",
    "transaction_dayofweek", "merchant_fraud_rate",
    "customer_avg_transaction_amount", "merchant_category_index",
    "device_type_index", "channel_grouped_index", "location_city_grouped_index"
]

# Loading production model (XGBoost)

client = MlflowClient()
registered_name = "workspace.ml_layer.Modelo_Fraude_XGB_Transactions"
versions       = client.search_model_versions(f"name='{registered_name}'")
latest_version = max([int(v.version) for v in versions])
xgb_model      = mlflow.xgboost.load_model(f"models:/{registered_name}/{latest_version}")
print(f"Loaded XGBoost model version {latest_version}")

# Sampling transactions from dataset

txn_test = spark.table("workspace.ml_layer.transaction_test_features") \
    .withColumn("is_fraud", col("is_fraud").cast("double")) \
    .withColumn("features_arr", vector_to_array("features"))

# Sample a pool of real fraud and legit cases
fraud_pool = txn_test.filter("is_fraud = 1").limit(500).toPandas()
legit_pool = txn_test.filter("is_fraud = 0").limit(500).toPandas()

fraud_X = np.array(fraud_pool["features_arr"].tolist())
legit_X = np.array(legit_pool["features_arr"].tolist())

# Score the pools to find CLEAR examples 
fraud_scores_pool = xgb_model.predict_proba(fraud_X)[:, 1]
legit_scores_pool = xgb_model.predict_proba(legit_X)[:, 1]

# Selecting cases based on highest fraud scored

fraud_sorted_idx = np.argsort(fraud_scores_pool)[::-1]   # highest fraud scores first
legit_sorted_idx = np.argsort(legit_scores_pool)         # lowest legit scores first

# Select didactic set:
selected = [
    # Clearest fraud (highest scoring real fraud case)
    {
        "X": fraud_X[fraud_sorted_idx[0]],
        "description": "Confirmed fraud pattern (high confidence)",
        "true_label": "FRAUD"
    },
    # Strong fraud (another high scorer)
    {
        "X": fraud_X[fraud_sorted_idx[5]],
        "description": "Suspicious transaction (strong fraud signal)",
        "true_label": "FRAUD"
    },
    # Borderline fraud (a fraud case the model is less sure about)
    {
        "X": fraud_X[fraud_sorted_idx[len(fraud_sorted_idx)//2]],
        "description": "Subtle fraud (ambiguous signal)",
        "true_label": "FRAUD"
    },
    # Clear legitimate (lowest scoring real legit case)
    {
        "X": legit_X[legit_sorted_idx[0]],
        "description": "Routine legitimate transaction",
        "true_label": "LEGIT"
    },
    # Another clear legitimate
    {
        "X": legit_X[legit_sorted_idx[10]],
        "description": "Standard customer purchase",
        "true_label": "LEGIT"
    }
]

X_demo       = np.array([s["X"] for s in selected])
descriptions = [s["description"] for s in selected]
true_labels  = [s["true_label"] for s in selected]

print(f"Selected 5 real transactions: 3 fraud, 2 legit")

# Scoring Demo Transactions

scores      = xgb_model.predict_proba(X_demo)[:, 1]
predictions = (scores >= production_threshold).astype(int)

def assign_risk_tier(score):
    if score >= 0.55:   return "🔴 CRITICAL"
    elif score >= 0.50: return "🟡 HIGH"
    elif score >= 0.45: return "🟠 MEDIUM"
    else:               return "🟢 LOW"

action_map = {
    "🔴 CRITICAL": "AUTO-BLOCK — freeze immediately",
    "🟡 HIGH"    : "MANUAL REVIEW — analyst within 1 hour",
    "🟠 MEDIUM"  : "ENHANCED MONITORING — next-day review",
    "🟢 LOW"     : "APPROVE — proceed normally"
}

# SHAP reason codes

explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_demo)
if isinstance(shap_values, list):
    shap_values = shap_values[1]

results = []
for i in range(len(X_demo)):
    risk_tier = assign_risk_tier(scores[i])

    feature_impacts = sorted(
        zip(TXN_FEATURE_NAMES, shap_values[i], X_demo[i]),
        key=lambda x: abs(x[1]), reverse=True
    )
    reasons = []
    for feat_name, shap_val, feat_val in feature_impacts[:3]:
        direction = "↑" if shap_val > 0 else "↓"
        reasons.append(f"{direction} {feat_name} ({feat_val:.2f})")

    # Check if model decision matches true label
    demo_threshold = 0.50
    model_says_fraud = scores[i] >= demo_threshold
    true_is_fraud    = true_labels[i] == "FRAUD"
    correct          = "✅" if model_says_fraud == true_is_fraud else "❌"

    results.append({
        "Scenario"    : descriptions[i],
        "True Label"  : true_labels[i],
        "Fraud Score" : f"{scores[i]:.4f}",
        "Risk Tier"   : risk_tier,
        "Decision"    : action_map[risk_tier],
        "Correct?"    : correct,
        "Reason 1"    : reasons[0],
        "Reason 2"    : reasons[1],
        "Reason 3"    : reasons[2]
    })

results_df = pd.DataFrame(results)

print("\n" + "="*100)
print("  LIVE FRAUD DETECTION DEMO — Real Transactions from Test Set")
print("="*100)
display(results_df)

# Visual Summary

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(12, 6))
colors = []
for tier in results_df["Risk Tier"]:
    if "CRITICAL" in tier:  colors.append("#DC2626")
    elif "HIGH" in tier:    colors.append("#F59E0B")
    elif "MEDIUM" in tier:  colors.append("#FBBF24")
    else:                   colors.append("#16A34A")

scores_float = [float(s) for s in results_df["Fraud Score"]]
bars = ax.barh(range(len(results_df)), scores_float,
               color=colors, edgecolor="#1F2937", linewidth=1.5)

ax.axvline(demo_threshold, color="black", linestyle="--", linewidth=2,
           label=f"Demo threshold: {demo_threshold:.3f}")
ax.set_yticks(range(len(results_df)))
ax.set_yticklabels([f"{i+1}. {d} [{true_labels[i]}]"
                    for i, d in enumerate(descriptions)], fontsize=10)
ax.set_xlabel("Fraud Score", fontsize=12, fontweight="bold")
ax.set_xlim(0, 1)
ax.set_title("Live Demo — Real Transaction Scoring", fontsize=14, fontweight="bold")
ax.legend(loc="lower right")
ax.grid(axis="x", alpha=0.3)

for i, bar in enumerate(bars):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            results_df.iloc[i]["Risk Tier"], va="center",
            fontsize=10, fontweight="bold")

plt.tight_layout()
plt.savefig("/tmp/live_demo_results.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n Demo completed")